In [19]:
import os
import numpy as np
import pandas as pd
from typing import Tuple
from numpy.linalg import svd, norm
from sklearn.preprocessing import StandardScaler
import requests

In [20]:
def compute_svd(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return U, s, Vt using economy SVD (full_matrices=False)."""
    U, s, Vt = svd(A, full_matrices=False)
    return U, s, Vt

In [21]:
def reconstruct_from_svd(U: np.ndarray, s: np.ndarray, Vt: np.ndarray, k: int | None = None) -> np.ndarray:
    """Reconstruct matrix. If k provided, use rank-k approximation."""
    if k is None:
        S = np.diag(s)
    else:
        S = np.diag(s[:k])
        U = U[:, :k]
        Vt = Vt[:k, :]
    return U @ S @ Vt

In [22]:
def report_reconstruction(A: np.ndarray, Arec: np.ndarray) -> None:
    err_fro = norm(A - Arec, ord='fro')
    rel_err = err_fro / (norm(A, ord='fro') + 1e-12)
    print(f"Frobenius error: {err_fro:.6f}, Relative error: {rel_err:.6%}")

In [23]:
def part_a(matrix: np.ndarray | list | None = None) -> None:
    """
    Compute SVD of a small matrix.
    Accepts a Python list or numpy array. Converts input to ndarray.
    """
    # ensure numpy array
    if matrix is None:
        matrix = np.array([[5, 3, 0], [4, 2, 0], [2, 5, 0]], dtype=float)
    else:
        matrix = np.asarray(matrix, dtype=float)

    A = matrix
    print("Part A: input matrix A:")
    print(A)
    U, s, Vt = compute_svd(A)
    print("\nSingular values:", s)
    Arec = reconstruct_from_svd(U, s, Vt)
    print("\nReconstructed A (using all singular values):")
    print(np.round(Arec, 6))
    report_reconstruction(A, Arec)
    # show rank-1 .. rank-r approximations
    r = min(A.shape)
    for k in range(1, r + 1):
        A_k = reconstruct_from_svd(U, s, Vt, k=k)
        print(f"\nRank-{k} approximation (first 6 entries):")
        print(np.round(A_k.flatten()[:6], 6))
        report_reconstruction(A, A_k)


In [24]:
def part_b() -> None:
    """
    Use the small user-movie ratings matrix from the exercise:
    Dhanya  5 3
    Ananya  4 2
    Mini    2 5
    """
    print("\nPart B: small ratings SVD")
    rows = ["Dhanya", "Ananya", "Mini"]
    cols = ["Movie1", "Movie2", "Movie3"]  # if a third movie doesn't exist, last column can be zeros
    # Based on provided snippet: two movie columns only. We'll construct a 3x2 matrix.
    A = np.array([[5, 3],
                  [4, 2],
                  [2, 5]], dtype=float)
    print("Ratings matrix (users x movies):")
    print(pd.DataFrame(A, index=rows, columns=["Movie1", "Movie2"]))
    U, s, Vt = compute_svd(A)
    print("\nSingular values:", s)
    # Full reconstruction
    Arec = reconstruct_from_svd(U, s, Vt)
    report_reconstruction(A, Arec)
    # Rank-1 and Rank-2 approximations
    for k in (1, 2):
        A_k = reconstruct_from_svd(U, s, Vt, k=k)
        print(f"\nRank-{k} approximation:")
        print(np.round(A_k, 4))
        report_reconstruction(A, A_k)
    # If you want to center by user mean (common in recommender systems)
    user_means = A.mean(axis=1, keepdims=True)
    A_centered = A - user_means
    print("\nUser-mean centered matrix:")
    print(np.round(A_centered, 4))
    Uc, sc, Vtc = compute_svd(A_centered)
    print("Singular values (centered):", sc)

In [25]:
def download_wine_csv(target_path: str) -> None:
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
    if requests is None:
        raise RuntimeError("requests module not available to download dataset.")
    print(f"Downloading {url} ...")
    r = requests.get(url)
    r.raise_for_status()
    # File uses semicolon delimiter
    with open(target_path, "wb") as f:
        f.write(r.content)
    print("Downloaded.")

In [26]:
def part_c(csv_path: str = "winequality_red.csv", k: int = 3) -> None:
    """
    Load winequality_red.csv, select feature columns (exclude 'quality'),
    standardize, compute SVD, print first 5 singular values,
    reconstruct, compute Rank-k approx and variance explained.
    """
    if not os.path.exists(csv_path):
        try:
            download_wine_csv(csv_path)
        except Exception as e:
            print(f"Could not download dataset automatically: {e}")
            return

    # the UCI CSV uses semicolons
    df = pd.read_csv(csv_path, sep=";")
    if 'quality' in df.columns:
        features = df.drop(columns=['quality'])
    else:
        # assume last column is quality if names missing
        features = df.iloc[:, :-1]

    X = features.values.astype(float)
    print(f"\nPart C: loaded {csv_path} with shape {X.shape}.")
    # Standardize: zero mean unit variance per feature
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    U, s, Vt = compute_svd(Xs)
    print("\nFirst 5 singular values:", np.round(s[:5], 6))

    # Reconstruct using all singular values and report error
    Xrec = reconstruct_from_svd(U, s, Vt)
    print("\nReconstruction error using all singular values:")
    report_reconstruction(Xs, Xrec)

    # Rank-k approximation
    Xk = reconstruct_from_svd(U, s, Vt, k=k)
    print(f"\nRank-{k} approximation error:")
    report_reconstruction(Xs, Xk)

    # Variance explained by singular values
    sv_sq = s ** 2
    total = sv_sq.sum()
    for j in (1, 2, 3, 5, k):
        explained = sv_sq[:j].sum() / total
        print(f"Variance explained by first {j} components: {explained:.4%}")

In [27]:
part_a([[3, 2, 2], [2, 3, -2]])

Part A: input matrix A:
[[ 3.  2.  2.]
 [ 2.  3. -2.]]

Singular values: [5. 3.]

Reconstructed A (using all singular values):
[[ 3.  2.  2.]
 [ 2.  3. -2.]]
Frobenius error: 0.000000, Relative error: 0.000000%

Rank-1 approximation (first 6 entries):
[2.5 2.5 0.  2.5 2.5 0. ]
Frobenius error: 3.000000, Relative error: 51.449576%

Rank-2 approximation (first 6 entries):
[ 3.  2.  2.  2.  3. -2.]
Frobenius error: 0.000000, Relative error: 0.000000%


In [28]:
part_b()


Part B: small ratings SVD
Ratings matrix (users x movies):
        Movie1  Movie2
Dhanya     5.0     3.0
Ananya     4.0     2.0
Mini       2.0     5.0

Singular values: [8.6420534  2.88355908]
Frobenius error: 0.000000, Relative error: 0.000000%

Rank-1 approximation:
[[4.2553 3.8279]
 [3.2054 2.8834]
 [3.5915 3.2307]]
Frobenius error: 2.883559, Relative error: 31.651173%

Rank-2 approximation:
[[5. 3.]
 [4. 2.]
 [2. 5.]]
Frobenius error: 0.000000, Relative error: 0.000000%

User-mean centered matrix:
[[ 1.  -1. ]
 [ 1.  -1. ]
 [-1.5  1.5]]
Singular values (centered): [2.91547595e+00 7.85046229e-17]


In [29]:
part_c(csv_path="winequality_red.csv", k=3)


Part C: loaded winequality_red.csv with shape (1599, 11).

First 5 singular values: [70.395403 55.493509 49.792761 44.044964 39.165138]

Reconstruction error using all singular values:
Frobenius error: 0.000000, Relative error: 0.000000%

Rank-3 approximation error:
Frobenius error: 84.110871, Relative error: 63.420776%
Variance explained by first 1 components: 28.1739%
Variance explained by first 2 components: 45.6822%
Variance explained by first 3 components: 59.7781%
Variance explained by first 5 components: 79.5283%
Variance explained by first 3 components: 59.7781%
